# C3 - Day 6: Model 2 - fine-tune Qwen2.5-1.5B to write restart summaries (QLoRA)
**AnchorAI | Component 3: Cognitive Checkpointing**

**Task.** Given a student's checkpoint (goal, completed work, working point, problem, next step, resources, open items) and the predicted difficulty, write a *restart summary*. The summary must

* use **only** what the student wrote (no new code, answers or advice - that is the job of C4),
* have a length that follows the difficulty (Low = short, Medium = paragraph, High = fuller with a list of open items),
* rewrite "I" as "you" and repeat a student's question as an open question, never answer it.

**Method.** QLoRA: the base model is loaded in 4-bit and only small LoRA adapter matrices (about 1% of the weights) are trained. This fits on a free Colab **T4 GPU** in a few minutes.

**Data.** `data/generated/restart_summary_{train,val}.jsonl`: 432 training and 48 validation examples that are **synthetic** (generated by `model2/generate_data.py` from invented IT-student scenarios). The test set uses *different task families* and is only used in notebook 07.

> **Runtime -> Change runtime type -> T4 GPU** before running. Upload the three `restart_summary_*.jsonl` files and `frame_vocab.json` to `MyDrive/C3/data/generated/` first.

## Step 0 - Check the GPU, install libraries, mount Drive

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU found. Use Runtime -> Change runtime type -> T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!pip -q install peft bitsandbytes accelerate

In [ ]:
import os, json, random, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/C3'
    print('Running in Colab - Drive mounted.')
except ImportError:
    BASE = os.environ.get('C3_BASE', r'C:\Users\user\OneDrive\Documents\AnchorAI\AnchorAI\components\c3_checkpointing')
    print('Running locally.')

GEN_DIR = os.path.join(BASE, 'data', 'generated')
OUTPUT_DIR = os.path.join(BASE, 'outputs')
ADAPTER_DIR = os.path.join(BASE, 'qwen_c3_lora')
os.makedirs(OUTPUT_DIR, exist_ok=True)
MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'

def read_jsonl(name):
    with open(os.path.join(GEN_DIR, name), encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

train_rows = read_jsonl('restart_summary_train.jsonl')
val_rows = read_jsonl('restart_summary_val.jsonl')
print(len(train_rows), 'training examples,', len(val_rows), 'validation examples')
ex = train_rows[0]
print('\n--- one example ---')
print(ex['messages'][1]['content'])
print('\n--- expected summary ---')
print(ex['messages'][2]['content'])

## Step 1 - Helper code (same code the local tests run)
`encode_row` turns one chat example into numbers and **masks the prompt**, so the model is trained only on the answer. `train` wraps the Hugging Face `Trainer`. `generate_batch` writes answers for a list of prompts.

In [ ]:
"""LoRA training and generation helpers for Model 2.

This file is used twice: the Colab notebooks include its source, and the local tests run the same code
on a tiny random model, so a mistake here is caught before any GPU time is spent.
"""
import inspect
import time

import torch
from torch.utils.data import Dataset
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments, set_seed

LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]


def encode_row(tok, row, max_len=1024):
    """Tokenise one chat example; only the assistant answer is trained on (the prompt is masked with -100)."""
    prompt = tok.apply_chat_template(row["messages"][:2], tokenize=False, add_generation_prompt=True)
    full = tok.apply_chat_template(row["messages"], tokenize=False)
    if not full.startswith(prompt):
        raise ValueError("chat template does not start with the prompt for " + row.get("id", "?"))
    prompt_ids = tok(prompt, add_special_tokens=False)["input_ids"]
    full_ids = tok(full, add_special_tokens=False)["input_ids"]
    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
    n = min(len(full_ids), max_len)
    return {"input_ids": full_ids[:n], "attention_mask": [1] * n, "labels": labels[:n]}


class ListDataset(Dataset):
    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return self.items[i]


def attach_lora(model, quantized, r=16, alpha=32, dropout=0.05):
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

    if quantized:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    config = LoraConfig(r=r, lora_alpha=alpha, lora_dropout=dropout, bias="none", task_type="CAUSAL_LM",
                        target_modules=LORA_TARGETS)
    return get_peft_model(model, config)


def training_arguments(output_dir, epochs, lr, batch_size, accum, has_eval, fp16, seed):
    kwargs = dict(
        output_dir=output_dir, num_train_epochs=epochs, per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size, gradient_accumulation_steps=accum, learning_rate=lr,
        lr_scheduler_type="cosine", warmup_steps=5, logging_steps=5, save_strategy="no", report_to="none",
        fp16=fp16, seed=seed, remove_unused_columns=False,
    )
    if has_eval:
        accepted = inspect.signature(TrainingArguments.__init__).parameters
        kwargs["eval_strategy" if "eval_strategy" in accepted else "evaluation_strategy"] = "epoch"
    return TrainingArguments(**kwargs)


def train(model, tok, train_rows, val_rows, out_dir, epochs=5, lr=2e-4, batch_size=4, accum=4, fp16=True,
          max_len=1024, seed=42):
    set_seed(seed)
    train_items = [encode_row(tok, r, max_len) for r in train_rows]
    val_items = [encode_row(tok, r, max_len) for r in val_rows]
    collator = DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100)
    args = training_arguments(out_dir, epochs, lr, batch_size, accum, bool(val_items), fp16, seed)
    trainer = Trainer(model=model, args=args, train_dataset=ListDataset(train_items),
                      eval_dataset=ListDataset(val_items) if val_items else None, data_collator=collator)
    model.config.use_cache = False
    start = time.time()
    result = trainer.train()
    model.config.use_cache = True
    info = {
        "seconds": round(time.time() - start, 1), "train_loss": float(result.training_loss),
        "epochs": epochs, "learning_rate": lr, "batch_size": batch_size, "grad_accumulation": accum,
        "max_len": max_len, "seed": seed, "n_train": len(train_items), "n_val": len(val_items),
        "longest_example_tokens": max(len(i["input_ids"]) for i in train_items),
        "history": trainer.state.log_history,
    }
    return trainer, info


@torch.no_grad()
def generate_batch(model, tok, message_lists, max_new_tokens=300, batch_size=8):
    """Greedy generation for a list of chat prompts (system + user messages); returns the answers."""
    model.eval()
    tok.padding_side = "left"
    answers = []
    for i in range(0, len(message_lists), batch_size):
        chunk = message_lists[i:i + batch_size]
        prompts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in chunk]
        enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
        for j in range(len(chunk)):
            new_tokens = out[j][enc["input_ids"].shape[1]:]
            answers.append(tok.decode(new_tokens, skip_special_tokens=True).strip())
    return answers


## Step 2 - Load the model in 4-bit and attach the LoRA adapters
`Qwen/Qwen2.5-1.5B-Instruct` is downloaded (about 3 GB the first time) and compressed to 4-bit on the fly. LoRA then adds small trainable matrices to the attention and feed-forward layers.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True,
                         bnb_4bit_compute_dtype=torch.float16)
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb, device_map='auto')
model = attach_lora(model, quantized=True)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable parameters: {trainable:,} of {total:,} ({100 * trainable / total:.2f}%)')

## Step 3 - Train
Settings: 5 passes over the data (epochs), learning rate 2e-4, batch of 4 with 4 accumulation steps (effective batch 16). The validation loss is computed after every epoch; if it keeps falling while the training loss falls, the model is learning the task and not just memorising.

In [ ]:
trainer, info = train(model, tok, train_rows, val_rows, os.path.join(BASE, 'qwen_c3_runs'),
                      epochs=5, lr=2e-4, batch_size=4, accum=4, fp16=True, max_len=1024, seed=42)
print(f"Done in {info['seconds']} s | mean training loss {info['train_loss']:.3f} | longest example {info['longest_example_tokens']} tokens")

In [ ]:
hist = info['history']
tl = [(h['step'], h['loss']) for h in hist if 'loss' in h]
vl = [(h['step'], h['eval_loss']) for h in hist if 'eval_loss' in h]
plt.figure(figsize=(7, 3.8))
plt.plot(*zip(*tl), label='training loss')
if vl:
    plt.plot(*zip(*vl), 'o-', label='validation loss')
plt.xlabel('step'); plt.ylabel('loss'); plt.title('Model 2 - QLoRA training'); plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'model2_training_loss.png'), dpi=150)
plt.show()
print('validation loss per epoch:', [round(v, 3) for _, v in vl])

## Step 4 - Save the adapter
Only the adapter (a few tens of MB) is saved, to Google Drive. It is **not** committed to git.

In [ ]:
model.save_pretrained(ADAPTER_DIR)
tok.save_pretrained(ADAPTER_DIR)
meta = {k: v for k, v in info.items() if k != 'history'}
meta.update({'base_model': MODEL_NAME, 'method': 'QLoRA (4-bit NF4, LoRA r=16, alpha=32, dropout=0.05)',
             'data': 'synthetic restart_summary_train.jsonl'})
with open(os.path.join(ADAPTER_DIR, 'training_info.json'), 'w') as f:
    json.dump(meta, f, indent=2)
print('Saved to', ADAPTER_DIR, sorted(os.listdir(ADAPTER_DIR)))
print(json.dumps(meta, indent=2))

## Step 5 - Quick look at three validation examples
Not an evaluation (that is notebook 07); just a sanity check that the model produces sensible summaries.

In [ ]:
sample = [r for lvl in ('Low', 'Medium', 'High') for r in val_rows if r['level'] == lvl][:1] + \
         [r for r in val_rows if r['level'] == 'Medium'][1:2] + [r for r in val_rows if r['level'] == 'High'][1:2]
answers = generate_batch(model, tok, [r['messages'][:2] for r in sample], max_new_tokens=300, batch_size=3)
for r, a in zip(sample, answers):
    print('=' * 80); print(r['level'], '|', r['family']); print(r['input']); print('--- model ---'); print(a)

## Summary - Day 6
* Training time, validation loss per epoch and the adapter location are printed above.
* The adapter is in `MyDrive/C3/qwen_c3_lora`; `training_info.json` records the settings.

**Next: notebook 07 - compare the fine-tuned model with the base model and with the template on the unseen test families.**